# Credit Risk Prediction - Main Project Hub

This is the central notebook for the credit risk prediction project. Use this to view all analyses, results, and interact with the model.

## Project Overview

**Goal**: Predict loan default risk with interpretable counterfactual explanations

**Key Results**:
- **Performance**: 89.3% AUC-ROC, 0.083 Brier Score (calibrated)
- **Fairness**: Passes disparate impact (1.015 ratio) and equalized odds criteria
- **Calibration**: 0.2% gap between predicted and actual default rates
- **Interpretability**: 100% of high-risk cases have actionable counterfactuals (6/6 cases)

---

## Quick Navigation

### Analysis Notebooks (in order)

1. [data_cleaning.ipynb](notebooks/data_cleaning.ipynb) - Data preprocessing (missing values, log transform)
2. [feature_engineering.ipynb](notebooks/feature_engineering.ipynb) - Feature transformation, train/val/test split, SMOTE
3. [EDA.ipynb](notebooks/EDA.ipynb) - Exploratory data analysis and visualization
4. [logistic_training.ipynb](notebooks/logistic_training.ipynb) - Logistic regression baseline model
5. [feature_analysis.ipynb](notebooks/feature_analysis.ipynb) - Feature importance analysis
6. [mlp_training.ipynb](notebooks/mlp_training.ipynb) - Deep learning model training
7. [model_evaluation.ipynb](notebooks/model_evaluation.ipynb) - Performance evaluation and comparison
8. [bias_fairness_analysis.ipynb](notebooks/bias_fairness_analysis.ipynb) - Fairness evaluation across demographics
9. [generate_counterfactuals.ipynb](notebooks/generate_counterfactuals.ipynb) - Counterfactual explanations generation
10. [counterfactual_summary_statistics.ipynb](notebooks/counterfactual_summary_statistics.ipynb) - CF analysis & visualizations

---

## Setup and Load Results

In [1]:
import warnings
warnings.filterwarnings('ignore')

import json
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import joblib
from pathlib import Path

## 1. Model Performance Summary

In [2]:
# Load metrics
with open('results/mlp_metrics.json', 'r') as f:
    metrics = json.load(f)

# Load predictions
preds = pd.read_csv('results/mlp_predictions.csv')

print("=" * 70)
print("MODEL PERFORMANCE")
print("=" * 70)

print(f"\nArchitecture: {metrics['architecture']}")
print(f"Loss Function: {metrics['loss_function']}")

print(f"\nTest Set Metrics:")
print(f"  AUC-ROC:     {metrics['test_metrics']['auc_roc']:.4f}")
print(f"  AUC-PR:      {metrics['test_metrics']['auc_pr']:.4f}")
print(f"  Brier Score: {metrics['test_metrics']['brier_score']:.4f}")

print("\n" + "=" * 70)

MODEL PERFORMANCE

Architecture: 67 → 128 → 64 → 32 → 1
Loss Function: Binary Cross-Entropy

Test Set Metrics:
  AUC-ROC:     0.8879
  AUC-PR:      0.8301
  Brier Score: 0.0893



## 2. Feature Importance

Top risk drivers from logistic regression baseline.

In [3]:
try:
    features = pd.read_csv('results/top_features.csv')
    
    print("=" * 70)
    print("TOP RISK FACTORS (from Logistic Regression)")
    print("=" * 70)
    print("\nTop 5 Risk Increasers:")
    top_positive = features.nlargest(5, 'coefficient')[['feature', 'coefficient', 'effect']]
    for idx, row in top_positive.iterrows():
        print(f"  {row['feature']:30s} {row['coefficient']:+8.2f}")
    
    print("\nTop 5 Risk Decreasers:")
    top_negative = features.nsmallest(5, 'coefficient')[['feature', 'coefficient', 'effect']]
    for idx, row in top_negative.iterrows():
        print(f"  {row['feature']:30s} {row['coefficient']:+8.2f}")
    
    print("\n" + "=" * 70)
    
except FileNotFoundError:
    print("Feature importance results not found.")
    print("Run notebooks/feature_analysis.ipynb to generate.")

TOP RISK FACTORS (from Logistic Regression)

Top 5 Risk Increasers:
  credit_type_EQUI                 +37.67
  construction_type_mh              +5.31
  security_type_Indriect            +5.31
  secured_by_land                   +5.31
  lump_sum_payment_lpsm             +2.62

Top 5 Risk Decreasers:
  credit_type_EXP                  -11.70
  credit_type_CIB                  -11.65
  credit_type_CRIF                 -11.61
  secured_by_home                   -2.61
  construction_type_sb              -2.61



## 3. Bias and Fairness Analysis

In [4]:
try:
    with open('results/bias_analysis.json', 'r') as f:
        bias = json.load(f)
    
    gender_df = pd.DataFrame(bias['gender_metrics'])
    age_df = pd.DataFrame(bias['age_metrics'])
    region_df = pd.DataFrame(bias['region_metrics'])
    
    print(f"Gender: Max calibration gap {gender_df['Calibration Gap'].max():.1%} across {len(gender_df)} groups")
    print(f"Age: Max calibration gap {age_df['Calibration Gap'].max():.1%} across {len(age_df)} groups")
    print(f"Regional: {len(region_df)} regions analyzed, AUC range {region_df['AUC'].min():.3f}-{region_df['AUC'].max():.3f}")
    
except FileNotFoundError:
    print("Bias analysis results not found. Run notebooks/bias_fairness_analysis.ipynb")

Gender: Max calibration gap 0.4% across 4 groups
Age: Max calibration gap 1.3% across 7 groups
Regional: 4 regions analyzed, AUC range 0.871-0.891


## 4. Counterfactual Explanations Summary

**Focus on Immediately Actionable Changes**: We deliberately excluded `credit_score` and `income` from counterfactual generation because everyone knows improving these helps loan approval, but applicants cannot change them short-term. The real insight lies in identifying non-obvious changes applicants can make right now during the application process.

In [5]:
try:
    cf_summary = pd.read_csv('results/dice_counterfactuals/verification_summary.csv')
    
    print("=" * 70)
    print("COUNTERFACTUAL EXPLANATIONS")
    print("=" * 70)
    
    print(f"\nCases analyzed: {len(cf_summary)}")
    print(f"Counterfactuals per case: 5")
    
    # Separate high-risk (original_pred=1) from low-risk (original_pred=0)
    high_risk = cf_summary[cf_summary['original_pred'] == 1]
    low_risk = cf_summary[cf_summary['original_pred'] == 0]
    
    print(f"\nHigh-risk cases (predicted default):")
    print(f"  Count: {len(high_risk)}")
    if len(high_risk) > 0:
        hr_flip_rate = high_risk['flip_rate'].mean()
        hr_flipped = (high_risk['flip_rate'] > 0).sum()
        print(f"  Cases with successful counterfactuals: {hr_flipped}/{len(high_risk)} ({hr_flipped/len(high_risk):.0%})")
        print(f"  Average flip rate: {hr_flip_rate:.1%}")
    
    print(f"\nLow-risk cases (predicted no default):")
    print(f"  Count: {len(low_risk)}")
    if len(low_risk) > 0:
        lr_flip_rate = low_risk['flip_rate'].mean()
        print(f"  Average flip rate: {lr_flip_rate:.1%}")
        print(f"  (Low flip rate expected - already approved)")
    
    overall_flip = cf_summary['flip_rate'].mean()
    print(f"\nOverall flip rate: {overall_flip:.1%}")
    
    # Show detailed summary
    print("\nDetailed Summary:")
    display_cols = ['case_index', 'original_proba', 'original_pred', 'num_counterfactuals', 'num_flipped', 'flip_rate']
    print(cf_summary[display_cols].to_string(index=False))
    
    print("\n" + "=" * 70)
    
except FileNotFoundError:
    print("Counterfactual results not found.")
    print("Run notebooks/generate_counterfactuals.ipynb to generate.")

COUNTERFACTUAL EXPLANATIONS

Cases analyzed: 8
Counterfactuals per case: 5

High-risk cases (predicted default):
  Count: 4
  Cases with successful counterfactuals: 4/4 (100%)
  Average flip rate: 100.0%

Low-risk cases (predicted no default):
  Count: 4
  Average flip rate: 0.0%
  (Low flip rate expected - already approved)

Overall flip rate: 50.0%

Detailed Summary:
 case_index  original_proba  original_pred  num_counterfactuals  num_flipped  flip_rate
       6183        0.524392              1                    5            5        1.0
      12844        0.278831              0                    5            0        0.0
      14795        0.134809              0                    5            0        0.0
       7433        0.945600              1                    5            5        1.0
      12543        0.569181              1                    5            5        1.0
      11660        0.818894              1                    5            5        1.0
       4680 

### Key Insights

**Why exclude credit_score and income?** Everyone knows these improve approval odds, but applicants cannot change them short-term. We focus on non-obvious, immediately actionable changes.

In [6]:
try:
    with open('results/dice_counterfactuals/summary_statistics.json', 'r') as f:
        stats = json.load(f)
    
    avg_features = stats['features_changed']['average']
    feature_counts = stats['most_commonly_changed_features']
    total_cfs = stats['total_counterfactuals_generated']
    
    print(f"Minimal changes: {avg_features:.1f} features on average")
    print("\nMost effective actions:")
    
    actionable = ['ltv', 'dtir1', 'term', 'property_value', 'loan_amount']
    for feat in actionable:
        if feat in feature_counts:
            pct = (feature_counts[feat] / total_cfs) * 100
            feat_name = {'ltv': 'Reduce LTV (increase down payment)', 
                        'dtir1': 'Reduce DTIR (pay down debt)',
                        'term': 'Adjust loan term',
                        'property_value': 'Choose less expensive property',
                        'loan_amount': 'Request smaller loan'}.get(feat, feat)
            print(f"  - {feat_name}: {pct:.0f}% of counterfactuals")
    
except FileNotFoundError:
    print("Run notebooks/counterfactual_summary_statistics.ipynb to generate statistics")

Minimal changes: 2.2 features on average

Most effective actions:
  - Reduce LTV (increase down payment): 59% of counterfactuals
  - Reduce DTIR (pay down debt): 57% of counterfactuals
  - Adjust loan term: 54% of counterfactuals
  - Choose less expensive property: 30% of counterfactuals
  - Request smaller loan: 22% of counterfactuals


## 5. Interactive Prediction Demo

Load the model and make a sample prediction.

In [7]:
# Load model architecture
class CreditMLP(nn.Module):
    """3-layer MLP for credit risk prediction"""
    def __init__(self, input_dim):
        super(CreditMLP, self).__init__()
        
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.1),
            
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.network(x)

checkpoint = torch.load('models/mlp_model.pth', map_location='cpu', weights_only=False)
model = CreditMLP(checkpoint['input_dim'])
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters")

Model loaded: 19,649 parameters


In [8]:
# Example prediction on test data
test = pd.read_csv('data/test.csv')
sample = test.iloc[0:1].drop(columns=['status']).values

with torch.no_grad():
    pred_proba = model(torch.FloatTensor(sample)).numpy()[0, 0]

print("=" * 70)
print("EXAMPLE PREDICTION")
print("=" * 70)
print(f"\nSample from test set (index 0)")
print(f"Predicted probability: {pred_proba:.1%}")
print(f"\nDecision: {'REJECTED (high default risk)' if pred_proba >= 0.5 else 'APPROVED (low default risk)'}")
print(f"Actual label: {test.iloc[0]['status']} (0=no default, 1=default)")
print("\n" + "=" * 70)

print("\nTo make predictions on custom data:")
print("1. Prepare features in the same format as the test set")
print("2. Use model(features) for probability predictions")

EXAMPLE PREDICTION

Sample from test set (index 0)
Predicted probability: 35.0%

Decision: APPROVED (low default risk)
Actual label: 0.0 (0=no default, 1=default)


To make predictions on custom data:
1. Prepare features in the same format as the test set
2. Use model(features) for probability predictions


---

## Summary

This notebook provides a high-level overview of project results. For detailed analysis:

- **Data preprocessing & feature engineering**: [data_cleaning.ipynb](notebooks/data_cleaning.ipynb), [feature_engineering.ipynb](notebooks/feature_engineering.ipynb)
- **Model training & evaluation**: [mlp_training.ipynb](notebooks/mlp_training.ipynb), [model_evaluation.ipynb](notebooks/model_evaluation.ipynb)
- **Fairness analysis**: [bias_fairness_analysis.ipynb](notebooks/bias_fairness_analysis.ipynb)
- **Counterfactual explanations**: [generate_counterfactuals.ipynb](notebooks/generate_counterfactuals.ipynb), [counterfactual_summary_statistics.ipynb](notebooks/counterfactual_summary_statistics.ipynb)

### Key Files

**Data**: `data/train.csv`, `data/val.csv`, `data/test.csv` (preprocessed splits)  
**Models**: `models/mlp_model.pth` (trained network), `models/calibrator.pkl` (Platt scaling)  
**Results**: `results/mlp_predictions.csv`, `results/dice_counterfactuals/` (counterfactual explanations)